# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides a step-by-step guide to loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset is defined by a Croissant schema and sourced from:
- [https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json)

**Note:** All entities are referenced by their `@id` field for accuracy and reproducibility.

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the FAIR^2 dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Print dataset name and description
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id` URIs.

### Listing Record Sets
We enumerate all record sets defined in the dataset, referencing by their `@id`.

In [ ]:
# List record sets and their @id
record_sets = dataset.metadata.recordSet or []

if not record_sets:
    print("No record sets detected in metadata; attempting autodiscovery via 'dataset.record_sets'.")
    # Try autodiscover using mlcroissant API
    record_sets = [x['@id'] for x in dataset.record_sets()]
else:
    # If available, retrieve @id values from metadata
    record_sets = [r['@id'] if isinstance(r, dict) and '@id' in r else r for r in record_sets]

print(f"Found {len(record_sets)} record sets:")
for rs_id in record_sets:
    print(f"- {rs_id}")

### Overview Example: Fields and Columns in Record Sets

Now, for each available record set, we'll review a sample (first 2 records), referencing by their `@id`.

**Note:** Replace `<record_set_id>` below with the actual `@id` from the previous cell. Entities such as fields and columns will always be referenced by their `@id`.

In [ ]:
for rs_id in record_sets:
    print(f"Sample records from record set {rs_id}:")
    for i, record in enumerate(dataset.records(record_set=rs_id)):
        print(record)
        if i >= 1:
            break
    print('\n')

## 3. Data Extraction

Load data from each record set into a Pandas DataFrame for analysis.
Reference record set and field `@id`s identified above.

In [ ]:
# Extract all record sets by @id
dataframes = {}

for rs_id in record_sets:
    records = list(dataset.records(record_set=rs_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"DataFrame for record set '{rs_id}' columns:")
        print(df.columns.tolist())
        print(df.head(2), '\n')
    else:
        print(f"No records loaded for record set '{rs_id}'.")

## 4. Exploratory Data Analysis (EDA)
Apply common processing steps such as filtering records, normalizing fields, and grouping by attributes, always referencing by `@id`.

**Example:** Select a numeric field, filter, normalize, and group by categorical field.

In [ ]:
# Choose a main record set (for demonstration, use the first available)
if dataframes:
    main_rs_id = list(dataframes.keys())[0]
    df = dataframes[main_rs_id]
else:
    raise ValueError("No dataframes loaded.")

# Review available fields/columns (@id)
print(f"Available columns in record set '{main_rs_id}':")
for col in df.columns:
    print(f"  - {col}")

# For demonstration, select a numeric field by its @id
numeric_fields = [c for c in df.columns if 'age' in c.lower() or 'interval' in c.lower() or 'years' in c.lower()]
if numeric_fields:
    numeric_field_id = numeric_fields[0]
else:
    # fallback: select any field likely to be numeric
    numeric_field_id = df.columns[0]

print(f"Selected numeric field for EDA: {numeric_field_id}")

# Filtering
threshold = 10
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold}:")
print(filtered_df.head())

# Normalization
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Grouping
group_fields = [c for c in df.columns if 'sex' in c.lower() or 'anatomical' in c.lower() or 'status' in c.lower() or 'location' in c.lower()]
if group_fields:
    group_field_id = group_fields[0]
    print(f"Grouping by field: {group_field_id}")
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"Grouped (mean) {numeric_field_id} by {group_field_id}:")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions and relationships between fields, always referencing fields by their `@id`.

We'll plot the distribution of the numeric field and visualize group differences.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of numeric field
plt.figure(figsize=(6,4))
sns.histplot(df[numeric_field_id], kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.show()

# If grouping field exists, boxplot
if group_fields:
    plt.figure(figsize=(8,4))
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.show()

## 6. Conclusion
In this notebook, we've:
- Loaded the FAIR^2 dataset using its Croissant schema with `mlcroissant`.
- Explored available record sets and fields via their `@id`.
- Extracted tabular data from key record sets and referenced columns using their IDs.
- Applied exploratory filtering, normalization, and grouping based on relevant `@id` fields.
- Visualized field distributions.

This approach demonstrates reproducible, schema-driven data exploration using the Croissant standard. For further analysis, you can extend EDA and modeling using only the specified `@id` URIs.